<div style="background: linear-gradient(135deg, #1a3a5c 0%, #2d6a9f 100%); padding: 40px 32px 32px 32px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: #ffffff; font-size: 2.0em; margin: 0 0 6px 0; font-family: 'Segoe UI', sans-serif; font-weight: 700;">
    Week 9, Lesson 16 -- Deep Learning with PyTorch
  </h1>
  <h2 style="color: #a8d4f5; font-size: 1.2em; margin: 0 0 18px 0; font-family: 'Segoe UI', sans-serif; font-weight: 400;">
    Tensors, Autograd, and Your First Training Loop From Scratch
  </h2>
  <hr style="border: 1px solid rgba(255,255,255,0.25); margin: 16px 0;">
  <table style="color: #cce4ff; font-family: 'Segoe UI', sans-serif; font-size: 0.95em;">
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Week:</strong></td><td>9 (Lesson 16) -- Deep Learning with PyTorch</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>Duration:</strong></td><td>40 Minutes</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Track:</strong></td><td>Petroleum Engineers &amp; Geoscientists</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>Audience:</strong></td><td>Engineers &amp; Geoscientists</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Instructor:</strong></td><td>Dr. Daniel Wamriew</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>Contact:</strong></td><td>wamriewdan@gmail.com</td>
    </tr>
  </table>
</div>


<div style="background: #e8f5e9; border-left: 5px solid #2ca87f; padding: 14px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### How to Use This Notebook

Work through this notebook **from top to bottom**, pressing **Shift + Enter** on each cell to run it.

- **Code cells** contain Python -- run them and read the output carefully.
- **Markdown cells** (white background) contain explanations -- read before running the next cell.
- **Student Activity** cells are marked with `Activity` -- complete these before moving on.
- **Homework** cells are marked with `Homework` -- complete after the session.
- You need **`well_log_data.csv`** in the course `data` folder (same file as Lesson 15).
- Install PyTorch once if needed: **`pip install torch`**

</div>


---
## Table of Contents

1. [From MLPClassifier to PyTorch: Why Go Lower-Level?](#1-from-mlpclassifier-to-pytorch-why-go-lower-level)
2. [Tensors: The Basic Building Block](#2-tensors-the-basic-building-block)
3. [Autograd: Automatic Differentiation](#3-autograd-automatic-differentiation)
4. [Prepare the Data](#4-prepare-the-data)
5. [Build a Network with torch.nn](#5-build-a-network-with-torchnn)
6. [Loss Functions](#6-loss-functions)
7. [Optimizers](#7-optimizers)
8. [The Training Loop](#8-the-training-loop)
9. [Evaluate: Predicting Permeability](#9-evaluate-predicting-permeability)
10. [PyTorch vs scikit-learn: What Changed?](#10-pytorch-vs-scikit-learn-what-changed)
11. [Student Activity](#11-student-activity)
12. [Recap & Homework](#12-recap-homework)


---
## 1. From MLPClassifier to PyTorch: Why Go Lower-Level?

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

In Lesson 13, one line built and trained a neural network:

```python
mlp = MLPClassifier(hidden_layer_sizes=(8,), activation="relu", max_iter=1000)
mlp.fit(X_train, y_train)
```

That is perfect for standard tabular problems. But scikit-learn hides every moving part: you
cannot change the loss function, add a physics constraint, process an image volume, or build an
architecture scikit-learn doesn't offer.

**PyTorch** removes the wrapper. You will build the same kind of network as Lesson 15, but now you
will see and control every step: the tensors, the forward pass, the loss, the gradients, and the
parameter update. This is the foundation for everything later in the course -- CNNs for seismic
images (Lesson 17+), sequence models for production forecasting, and physics-informed networks.

Today's target: predict **porosity** (a continuous number) from well log features -- a
regression problem, so we'll pair PyTorch with mean-squared-error loss instead of Lesson 15's
classification setup.

</div>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


---
## 2. Tensors: The Basic Building Block

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

A **tensor** is PyTorch's version of a NumPy array -- a grid of numbers with a shape (dimensions).
The difference: tensors can track gradients and run on a GPU. If you know NumPy, you already know
most of the tensor API.

| NumPy | PyTorch |
|---|---|
| `np.array([1, 2, 3])` | `torch.tensor([1, 2, 3])` |
| `arr.shape` | `tensor.shape` |
| `arr.reshape(2, 3)` | `tensor.reshape(2, 3)` / `tensor.view(2, 3)` |
| `arr @ other` | `tensor @ other` |
| `arr.mean()` | `tensor.mean()` |

</div>


In [ ]:
# A 1D tensor -- three scaled log values, same numbers from Lesson 13's toy neuron
x = torch.tensor([0.8, -0.3, 1.1])
print("x:", x)
print("shape:", x.shape, "  dtype:", x.dtype)

# A 2D tensor -- a tiny batch of 4 samples, 3 features each (rows = samples)
X_demo = torch.tensor([[0.8, -0.3, 1.1],
                       [0.2,  0.5, -0.4],
                       [-1.1, 0.9,  0.3],
                       [0.4, -0.7,  0.6]])
print("\nX_demo shape:", X_demo.shape)

# NumPy <-> PyTorch conversion, used constantly when moving data in and out
X_np = X_demo.numpy()
X_back = torch.from_numpy(X_np)
print("Round-trip matches:", torch.equal(X_demo, X_back))


---
## 3. Autograd: Automatic Differentiation

<div style="background: #fff8e1; border-left: 5px solid #e0a800; padding: 14px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

Lesson 13's `MLPClassifier.fit()` used **backpropagation** internally to compute how each weight
should change. PyTorch exposes this machinery directly through **autograd**.

Set `requires_grad=True` on a tensor, and PyTorch silently builds a computational graph as you do
math with it. Call `.backward()` on a final scalar result, and PyTorch fills in `.grad` on every
tensor upstream -- the exact partial derivative of the result with respect to that tensor.

This is the same chain rule from Lesson 13 (Section 3), just computed automatically instead of by
hand.

</div>


In [ ]:
# A minimal autograd example: z = w*x + b, loss = (z - target)**2
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(-0.1, requires_grad=True)
x_val = torch.tensor(2.0)
target = torch.tensor(1.5)

z = w * x_val + b
loss = (z - target) ** 2

loss.backward()  # computes d(loss)/dw and d(loss)/db automatically

print(f"z = {z.item():.3f},  loss = {loss.item():.3f}")
print(f"d(loss)/dw = {w.grad.item():.3f}")
print(f"d(loss)/db = {b.grad.item():.3f}")
print()
print("This single call replaces the hand-derived chain-rule steps from Lesson 15, Section 3 --")
print("and scales to networks with millions of parameters exactly the same way.")


---
## 4. Prepare the Data

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

Same dataset and same scaling discipline as Lesson 15 (Section 6): neural networks need scaled
inputs. Today's target is **porosity (`NPHI_frac`)**, predicted from the other logs -- the same
"predicting reservoir properties" task named in the course syllabus for this lesson.

The raw file has missing values in every log column (a realistic touch -- tool dropouts and washed-out
hole sections are common), so a cleaning step comes first, exactly like Lesson 3.

</div>


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/well_log_data.csv")
print(df.shape)
df.head()


In [ ]:
print("Missing values per column:")
print(df.isna().sum())

# Drop rows missing any of the columns we need today (Lesson 3 approach)
model_cols = ["GR_API", "RHOB_gcc", "RT_ohmm", "SW_frac", "NPHI_frac"]
df_clean = df.dropna(subset=model_cols).reset_index(drop=True)

print(f"\nRows before cleaning: {len(df)}")
print(f"Rows after cleaning:  {len(df_clean)}")


In [ ]:
# Resistivity spans orders of magnitude, so log10-transform it before it becomes a feature --
# the same reasoning used for permeability in petrophysics, applied here to RT_ohmm.
df_clean["logRT"] = np.log10(df_clean["RT_ohmm"])

features = ["GR_API", "RHOB_gcc", "logRT", "SW_frac"]
target = "NPHI_frac"

X = df_clean[features].values
y = df_clean[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Convert to PyTorch tensors -- float32 is the standard dtype for network weights
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("Train tensor shapes:", X_train_t.shape, y_train_t.shape)
print("Test tensor shapes: ", X_test_t.shape, y_test_t.shape)


> **Why `.view(-1, 1)`?** PyTorch's loss functions expect predictions and targets to have matching
> shapes. Our targets start as a flat 1D array; `view(-1, 1)` reshapes to a column, `(n_samples, 1)`,
> matching the shape the network will output.


---
## 5. Build a Network with torch.nn

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

`nn.Sequential` chains layers exactly like the diagram from Lesson 15, Section 5: an input layer,
one hidden layer with an activation, and an output layer. Each `nn.Linear(in, out)` is a full layer
of neurons -- it holds its own weight matrix and bias vector, initialized randomly, exactly like
`MLPClassifier` does internally.

</div>


In [ ]:
n_features = X_train_t.shape[1]

model = nn.Sequential(
    nn.Linear(n_features, 16),  # input layer -> hidden layer (16 neurons)
    nn.ReLU(),                  # activation function (Lesson 15, Section 4)
    nn.Linear(16, 8),           # hidden layer -> second hidden layer (8 neurons)
    nn.ReLU(),
    nn.Linear(8, 1)             # hidden layer -> output layer (1 neuron: predicted log-perm)
)

print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {n_params}")


> Compare this to Lesson 15's `MLPClassifier(hidden_layer_sizes=(8,))`. That one line built and
> trained a network in one step. Here, `nn.Sequential` only **defines the architecture** -- weights
> are randomly initialized but untrained. Training is a separate, explicit step, coming up in
> Section 8.


---
## 6. Loss Functions

<div style="background: #fff8e1; border-left: 5px solid #e0a800; padding: 14px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

The **loss function** measures how wrong a prediction is. Lesson 15's classifier minimized a
classification loss internally; today's regression problem uses **Mean Squared Error (MSE)**:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

PyTorch provides this as `nn.MSELoss()`. Other common choices you'll meet later: `nn.CrossEntropyLoss`
(classification), `nn.L1Loss` (mean absolute error, more robust to outliers).

</div>


In [ ]:
loss_fn = nn.MSELoss()

# Sanity check: loss on the untrained model
with torch.no_grad():
    y_pred_untrained = model(X_train_t)
    initial_loss = loss_fn(y_pred_untrained, y_train_t)

print(f"MSE loss before any training: {initial_loss.item():.4f}")
print("This should fall substantially once we train in Section 8.")


---
## 7. Optimizers

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

The **optimizer** takes the gradients computed by autograd and decides how to update each weight.
The simplest rule, gradient descent, is:

$$w \leftarrow w - \alpha \frac{\partial \, \text{loss}}{\partial w}$$

where $\alpha$ is the **learning rate**. PyTorch's `Adam` optimizer is a more robust, adaptive
version of this rule and is the standard default for most problems.

</div>


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(optimizer)
print("\n'model.parameters()' hands the optimizer every weight and bias in the network --")
print("this is the entire list of numbers it is allowed to adjust.")


---
## 8. The Training Loop

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

Every PyTorch training loop repeats the same four steps, once per epoch:

```
1. Forward pass   -- compute predictions:      y_pred = model(X)
2. Compute loss   -- compare to targets:       loss = loss_fn(y_pred, y)
3. Backward pass  -- compute gradients:        loss.backward()
4. Update weights -- take an optimizer step:   optimizer.step()
```

One extra housekeeping call, `optimizer.zero_grad()`, clears old gradients before each backward
pass -- PyTorch accumulates gradients by default, so forgetting this is the most common PyTorch bug.

</div>


In [ ]:
n_epochs = 300
train_losses = []

for epoch in range(n_epochs):
    model.train()

    optimizer.zero_grad()                     # 0. clear old gradients
    y_pred = model(X_train_t)                 # 1. forward pass
    loss = loss_fn(y_pred, y_train_t)         # 2. compute loss
    loss.backward()                           # 3. backward pass (autograd)
    optimizer.step()                          # 4. update weights

    train_losses.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:4d}/{n_epochs}   Training MSE: {loss.item():.4f}")

print("\nTraining complete.")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(train_losses, color="#2d6a9f", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training MSE Loss")
ax.set_title("Loss Curve -- Watching the Network Learn")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


<div style="background: #fff8e1; border-left: 5px solid #e0a800; padding: 14px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### What to Look For

A healthy loss curve drops quickly at first, then flattens. If it flattens too early at a high
value, the network is **underfitting** (Lesson 4) -- try more epochs, a higher learning rate, or a
bigger hidden layer. If it oscillates wildly, the learning rate is likely too high.

</div>


---
## 9. Evaluate: Predicting Permeability

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

`model.eval()` switches off training-only behavior (irrelevant for this simple network, but always
good practice), and `torch.no_grad()` tells PyTorch not to bother building a gradient graph since we
are only predicting, not training.

</div>


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

model.eval()
with torch.no_grad():
    y_pred_test_t = model(X_test_t)

y_pred_test = y_pred_test_t.numpy().flatten()
y_test_flat = y_test_t.numpy().flatten()

test_mse = mean_squared_error(y_test_flat, y_pred_test)
test_r2 = r2_score(y_test_flat, y_pred_test)

print(f"Test MSE (porosity, fraction): {test_mse:.5f}")
print(f"Test R-squared:                {test_r2:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_flat, y_pred_test, alpha=0.6, color="#2d6a9f", edgecolor="white", linewidth=0.3)

lims = [min(y_test_flat.min(), y_pred_test.min()), max(y_test_flat.max(), y_pred_test.max())]
ax.plot(lims, lims, color="#e07b39", linestyle="--", linewidth=2, label="Perfect prediction")

ax.set_xlabel("Actual Porosity (NPHI, fraction)")
ax.set_ylabel("Predicted Porosity (NPHI, fraction)")
ax.set_title(f"PyTorch Network -- Test Set (R\u00b2 = {test_r2:.3f})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
## 10. PyTorch vs scikit-learn: What Changed?

<div style="background: #e8f5e9; border-left: 5px solid #2ca87f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

| Step | scikit-learn (Lesson 13) | PyTorch (today) |
|---|---|---|
| Define architecture | `hidden_layer_sizes=(8,)` argument | `nn.Sequential(nn.Linear(...), nn.ReLU(), ...)` |
| Loss function | Fixed by the estimator class | Chosen explicitly: `nn.MSELoss()` |
| Optimizer | Fixed default (`adam`) | Chosen explicitly: `torch.optim.Adam(...)` |
| Training | One call: `.fit(X, y)` | Explicit loop: forward, loss, backward, step |
| Control | Low -- a handful of hyperparameters | High -- every computation is visible and editable |

Nothing about the underlying mathematics changed -- it is still weighted sums, activation functions,
and gradient descent. What changed is **who is in control of each step**. That control is exactly
what Weeks 8-10 need: custom architectures for images (CNNs) and sequences (LSTMs) that no
`MLPClassifier` argument can express.

</div>


---
## 11. Student Activity

<div style="background: #fff3cd; border-left: 5px solid #ffc107; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### Activity: Change the Learning Rate

Retrain the Section 8 model (rebuild `model` and `optimizer` fresh each time, so training starts
from scratch) with `lr=0.1` and then `lr=0.0001`, keeping `n_epochs=300`.

| Learning rate | Final training MSE | Test R² | Comment |
|---|---|---|---|
| 0.0001 | | | |
| 0.01 (baseline) | | | |
| 0.1 | | | |

**Reflection:** What does a learning rate that is too high do to the loss curve? Too low?

</div>


In [ ]:
# Activity -- rebuild model + optimizer, retrain with lr=0.1, then lr=0.0001, compare
# Hint: you MUST create a new nn.Sequential(...) each time -- reusing 'model' continues
# training from where it left off instead of starting fresh.



---
## 12. Recap & Homework

<div style="background: #e8f5e9; border-left: 5px solid #2ca87f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### What You Learned

| Concept | Meaning |
|---------|---------|
| Tensor | PyTorch's array type; like NumPy, but tracks gradients and can run on GPU |
| Autograd | Automatic computation of gradients via `.backward()` |
| `nn.Sequential` | Chains layers into a network architecture |
| Loss function | Measures prediction error, e.g. `nn.MSELoss()` for regression |
| Optimizer | Updates weights using gradients, e.g. `torch.optim.Adam` |
| Training loop | Forward pass -> loss -> backward pass -> optimizer step, repeated per epoch |

You have now built a neural network from its actual moving parts, not a single library call. Next,
Lesson 17 moves from tabular well logs to **images**: Convolutional Neural Networks (CNNs) for
seismic facies classification and fault detection.

</div>


### Homework

**Task 1 -- Depth Comparison**

Rebuild the model with only one hidden layer (`nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, 1)`)
and retrain from scratch. Compare test R² against the two-hidden-layer baseline from Section 8.
Does the extra layer help on this dataset?

**Task 2 -- Loss Function Swap**

Retrain using `nn.L1Loss()` instead of `nn.MSELoss()` (keep the architecture and optimizer the same).
Report test MSE and R² for both. Which loss gives a better R², and can you explain why L1 loss
might behave differently near outlier permeability values?

**Task 3 (Engineering) -- Predict a Different Target**

Repeat Section 4-9 predicting `SW_frac` (water saturation) instead of porosity, using `GR_API`,
`RHOB_gcc`, `NPHI_frac`, and `logRT` as features. Report test R² and compare briefly to how well
porosity was predicted -- which target is easier for the network, and why might that be?


In [ ]:
# Homework Task 1 -- Depth Comparison
# Hint: rebuild model = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, 1))
#       and a fresh optimizer, then reuse the Section 8 training loop.



In [ ]:
# Homework Task 2 -- Loss Function Swap
# Hint: loss_fn = nn.L1Loss(), fresh model + optimizer, reuse the Section 8 training loop.



In [ ]:
# Homework Task 3 (Engineering) -- Predict a Different Target
# Hint: features = ["GR_API", "RHOB_gcc", "NPHI_frac", "logRT"], target = df_clean["SW_frac"].values
# Rebuild train/test split, scaler, tensors, model, and optimizer, then retrain.



<div style="background: #1a3a5c; color: white; padding: 22px 26px; border-radius: 10px; margin-top: 18px;">
  <h3 style="margin-top: 0; color: #ffffff;">Lesson Complete</h3>
  <p style="margin-bottom: 0; color: #d8ecff;">
    You have built, trained, and evaluated a PyTorch neural network from its raw components --
    tensors, autograd, a training loop, a loss function, and an optimizer. Next week, Lesson 17
    opens with <strong>Convolutional Neural Networks</strong>: how these same building blocks are
    rearranged to read seismic images and detect faults directly from pixel data.
  </p>
</div>
